In [1]:
import yt_dlp
import os
import re

def extract_text_from_vtt(vtt_file_path):
    """
    Extrai o texto de um arquivo .vtt, removendo as marcações de tempo, formatação e duplicatas.
    """
    if not os.path.exists(vtt_file_path):
        return None

    subtitle_text = []
    with open(vtt_file_path, 'r', encoding='utf-8') as f:
        previous_line = None
        for line in f:
            # Remove as marcações de tempo e formatação usando expressão regular
            cleaned_line = re.sub(r'<[^>]+>', '', line).strip()
            # Ignora linhas vazias e continua adicionando o texto limpo
            if cleaned_line and not '-->' in cleaned_line:
                # Evita adicionar linhas duplicadas consecutivamente
                if cleaned_line != previous_line:
                    subtitle_text.append(cleaned_line)
                previous_line = cleaned_line
    
    # Junta o texto das legendas em uma única string, removendo possíveis duplicatas
    subtitle_text = "\n".join(subtitle_text)
    
    # Remover duplicatas em linhas não consecutivas
    unique_lines = list(dict.fromkeys(subtitle_text.splitlines()))
    return "\n".join(unique_lines)

def download_video_and_subtitles(url):
    try:
        # Opções de download para vídeo e legendas
        ydl_opts = {
            'outtmpl': 'downloaded_video.%(ext)s',  # Nome do arquivo de saída
            'format': 'bestvideo+bestaudio/best',  # Tenta baixar o melhor vídeo e áudio disponíveis
            'subtitleslangs': ['en'],  # Baixa as legendas em inglês
            'writesubtitles': True,  # Especifica que queremos as legendas
            'writeautomaticsub': True,  # Tenta pegar as legendas automáticas, se disponíveis
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info_dict = ydl.extract_info(url, download=True)
            formats = info_dict.get('formats', [])
            for f in formats:
                print(f"{f['format_id']}: {f['ext']} - {f.get('format_note', '')} - {f.get('resolution', '')}")
            video_filename = ydl.prepare_filename(info_dict)

            # Verifica se as legendas foram baixadas
            subtitle_filename = video_filename.rsplit('.', 1)[0] + ".en.vtt"
            subtitle_text = None
            if os.path.exists(subtitle_filename):
                subtitle_text = extract_text_from_vtt(subtitle_filename)

            return video_filename, subtitle_text

    except Exception as e:
        print(f"Failed to download video or subtitles: {str(e)}")
        return None, None

# Exemplo de uso:
url = 'https://www.youtube.com/watch?v=5xbADDvciko'
video_filename, subtitles_text = download_video_and_subtitles(url)

if video_filename:
    print(f"Video downloaded: {video_filename}")
    if subtitles_text:
        print("Subtitles text extracted:")
        print(subtitles_text)
    else:
        print("No subtitles found.")
else:
    print("Failed to download video.")

[youtube] Extracting URL: https://www.youtube.com/watch?v=5xbADDvciko
[youtube] 5xbADDvciko: Downloading webpage
[youtube] 5xbADDvciko: Downloading ios player API JSON
[youtube] 5xbADDvciko: Downloading mweb player API JSON
[youtube] 5xbADDvciko: Downloading player a62d836d
[youtube] 5xbADDvciko: Downloading m3u8 information
[info] 5xbADDvciko: Downloading subtitles: en
[info] 5xbADDvciko: Downloading 1 format(s): 616+251
Deleting existing file downloaded_video.en.vtt
[info] Writing video subtitles to: downloaded_video.en.vtt
[download] Destination: downloaded_video.en.vtt
[download] 100% of  554.26KiB in 00:00:00 at 2.14MiB/s
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 690
[download] Destination: downloaded_video.f616.mp4
[download] 100% of  741.03MiB in 00:03:38 at 3.39MiB/s                      
[download] Destination: downloaded_video.f251.webm
[download] 100% of   57.53MiB in 00:00:02 at 19.92MiB/s    
[Merger] Merging formats into "downloaded_video.webm"
De